In [12]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline

# Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC

# Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, f1_score
import time

import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from rital_nlp_project.common.utils import load_clean_data
from rital_nlp_project.common.notebook_utils import init_notebook
from rital_nlp_project.common.models_eval import eval_pipeline, eval_combination_matrix

from rital_nlp_project.presidents.utils import concat_seq_by_class

init_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
texts_pres, classes_pres = load_clean_data("../Dataset/presidents_clean.parquet")

In [ ]:
max_features = 5000
min_df = 5
max_df = 0.5

vectorizer_list = [
    CountVectorizer(analyzer="char", ngram_range=(1, 1), max_features=max_features),
    CountVectorizer(analyzer="char_wb", ngram_range=(4, 4), max_features=max_features, min_df=min_df, max_df=max_df),
    CountVectorizer(analyzer="word", ngram_range=(1, 1), max_features=max_features, min_df=min_df, max_df=max_df),
    CountVectorizer(analyzer="word", ngram_range=(1, 3), max_features=max_features, min_df=min_df, max_df=max_df),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(4, 4), max_features=max_features, min_df=min_df, max_df=max_df),

    TfidfVectorizer(analyzer="word", ngram_range=(1, 1), max_features=max_features, min_df=min_df, max_df=max_df),
    TfidfVectorizer(analyzer="word", ngram_range=(1, 3), max_features=max_features, min_df=min_df, max_df=max_df),
]

classifier_list = [
    LogisticRegression(max_iter=100),
    MultinomialNB(), 
    LinearSVC(),
    #SVC(kernel="rbf")
]

### Separated phrases

In [6]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(texts_pres, classes_pres, random_state=42)

In [ ]:
results = eval_combination_matrix(X_train, y_train, X_test, y_test, vectorizer_list, classifier_list)
results.to_csv("out/baseline_eval_matrix_pres.csv", index=False)

Pipeline(steps=[('vectorizer',
                 CountVectorizer(analyzer='char', max_features=5000)),
                ('classifier', LogisticRegression())])


/Users/bsh2022/Study/master_ue/rital/RITAL-NLP-project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/bsh2022/Study/master_ue/rital/RITAL-NLP-project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the d

Pipeline(steps=[('vectorizer',
                 CountVectorizer(analyzer='char', max_features=5000)),
                ('classifier', MultinomialNB())])
Pipeline(steps=[('vectorizer',
                 CountVectorizer(analyzer='char', max_features=5000)),
                ('classifier', LinearSVC())])
Pipeline(steps=[('vectorizer',
                 CountVectorizer(analyzer='char', max_features=5000)),
                ('classifier', SVC())])


In [4]:
# sns.heatmap(results.pivot(columns="classifier", index="vectorizer", values="f1"), annot=True, cmap="viridis")
# plt.title("F1-score")

In [8]:

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(analyzer="char", ngram_range=(1, 1))),
    ("classifier", MultinomialNB()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(analyzer="word", ngram_range=(1, 1), min_df=min_df, max_df=max_df, max_features=max_features)),
    ("classifier", MultinomialNB()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(analyzer="word", ngram_range=(1, 3), min_df=min_df, max_df=max_df, max_features=max_features)),
    ("classifier", MultinomialNB()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

Pipeline steps:
  vectorizer: CountVectorizer(analyzer='char')
  classifier: MultinomialNB()
Scores : [0.92933192 0.92911818 0.92916719 0.92928287 0.9295231 ], mean score : 0.9292846503573472
Training time: 0.377 seconds
Inference time: 0.122 seconds
Vocabulary size: 36
Train/test accuracy score: 0.9322210636079249
              precision    recall  f1-score   support

          -1       0.55      0.01      0.02      1823
           1       0.87      1.00      0.93     12531

    accuracy                           0.87     14354
   macro avg       0.71      0.50      0.48     14354
weighted avg       0.83      0.87      0.82     14354


Pipeline steps:
  vectorizer: CountVectorizer(max_df=0.5, max_features=5000, min_df=5)
  classifier: MultinomialNB()
Scores : [0.93129368 0.93148698 0.93152856 0.93949802 0.93350079], mean score : 0.9334616057668876
Training time: 0.198 seconds
Inference time: 0.064 seconds
Vocabulary size: 5000
Train/test accuracy score: 0.9355322338830585
            

In [24]:

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(analyzer="char", ngram_range=(1, 1))),
    ("classifier", LinearSVC()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(analyzer="word", ngram_range=(1, 1), min_df=min_df, max_df=max_df, max_features=max_features)),
    ("classifier", LinearSVC()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(analyzer="word", ngram_range=(1, 3), min_df=min_df, max_df=max_df, max_features=max_features)),
    ("classifier", LinearSVC()) 
])
eval_pipeline(X_train, y_train, X_test, y_test, pipeline, True)

Pipeline steps:
  vectorizer: TfidfVectorizer(analyzer='char')
  classifier: LinearSVC()
Scores : [0.7839784  0.79660259 0.7676951  0.7826087  0.7747106 ], mean score : 0.7811190765369669
Training time: 0.329 seconds
Inference time: 0.123 seconds
Vocabulary size: 36
Train/test accuracy score: 0.5562637362637363
              precision    recall  f1-score   support

          -1       0.14      0.67      0.23      1823
           1       0.89      0.40      0.56     12531

    accuracy                           0.44     14354
   macro avg       0.52      0.54      0.39     14354
weighted avg       0.80      0.44      0.52     14354


Pipeline steps:
  vectorizer: TfidfVectorizer(max_df=0.5, max_features=5000, min_df=5)
  classifier: LinearSVC()
Scores : [0.84904688 0.88077118 0.86292835 0.89026915 0.87213449], mean score : 0.8710300106535842
Training time: 0.207 seconds
Inference time: 0.073 seconds
Vocabulary size: 5000
Train/test accuracy score: 0.6144019217120708
              precis